# Sesgo por hora, mes por mes

`09_estaciones_criticas.ipynb` mostró que la sobreestimación no es de un tipo de estación sino de una **franja horaria**: en el conjunto de prueba principal (28 de julio a 31 de agosto), XGBoost predice de más de 19 a 23 h (de 13 % a 17 % de la demanda a las 20, 21 y 22 h) y de menos en el pico de la mañana (8 y 9 h).

Pregunta de este notebook: **¿ese patrón aparece en todos los meses o solo en el de prueba?** Se repite la validación mes por mes de `06_validacion_mensual.ipynb` (cada mes entrena con todo lo anterior, mismos hiperparámetros) guardando el sesgo **por hora** de cada mes, para XGBoost, el promedio histórico y la media de 4 semanas.

**Qué horas se miran.** De 6 a 23 h. El sistema está cerrado de las 00:30 a las 05:00 y casi no hay demanda: el sesgo relativo (sesgo entre demanda media de la hora) no tiene sentido ahí. Las horas 20 a 22 sí están dentro del horario de servicio, así que el cierre no explica por sí solo la sobreestimación de esas horas; si el cierre influye (por ejemplo, adelantando la salida de usuarios que prefieren llegar antes de que cierre) es una hipótesis que este notebook no prueba.

**Cómo correrlo:** Restart & Run All. Necesita `trips_clean.parquet` y `data/processed/busqueda_hiperparametros.csv`. Tarda lo mismo que `06` (siete entrenamientos de XGBoost) y necesita varios GB de RAM libres, igual que `06`. Guarda `resultados_sesgo_hora_mes.csv` después de cada mes y, si se interrumpe, retoma donde se quedó.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")
import modelo as M

DESDE = "2026-03-01"          # primer mes que se evalúa (si hay historia suficiente)
MIN_DIAS_TRAIN = 42
HORAS = list(range(6, 24))    # horas con servicio y demanda apreciable
SOLO_MESES = None             # por ejemplo ["2026-08"] para correr un solo mes
SALIDA = "../data/processed/resultados_sesgo_hora_mes.csv"
REHACER = False

# Igual que 06: si están los CSV de sep 2025 – feb 2026 en el parquet, se usan para entrenar los primeros meses
fechas_viajes = pd.read_parquet(f"{M.PROCESSED_DIR}/trips_clean.parquet", columns=["origen_dt"])["origen_dt"]
previos = ((fechas_viajes >= "2025-09-01") & (fechas_viajes < "2026-03-01")).sum()
M.FECHA_INICIO = "2025-09-01" if previos > 1_000_000 else "2026-03-01"
M.FECHA_FIN = "2026-09-30"
del fechas_viajes
print(f"Historia desde {M.FECHA_INICIO}")

In [ ]:
validos = M.cargar_validos()
estaciones = M.cargar_estaciones()
tabla = M.construir_tabla(validos, estaciones)
del validos
print(f"Filas: {len(tabla):,} | {tabla['fecha'].min().date()} a {tabla['fecha'].max().date()}")

h = pd.read_csv("../data/processed/busqueda_hiperparametros.csv").iloc[0]
params = {k: (h[k] if k == "objective" else int(h[k]) if k in ("max_depth", "min_child_weight") else float(h[k]))
          for k in M.ESPACIO_BUSQUEDA}
print("Hiperparámetros (versión 1):", params)

## 1. Entrenar mes por mes y guardar el sesgo por hora

Para cada hora se guarda la demanda media real, el sesgo (predicho menos real, en salidas por estación-hora) y el sesgo como porcentaje de esa demanda.

In [ ]:
def por_hora(test, pred):
    real = test[M.TARGET].to_numpy(dtype=float)
    d = pd.DataFrame({"hora": test["hora"].to_numpy(), "real": real, "err": np.asarray(pred, dtype=float) - real})
    g = d.groupby("hora")
    out = pd.DataFrame({"demanda": g["real"].mean(), "sesgo": g["err"].mean()})
    out["sesgo_rel_%"] = 100 * out["sesgo"] / out["demanda"].replace(0, np.nan)
    return out.reset_index()


previos_res = pd.read_csv(SALIDA) if (os.path.exists(SALIDA) and not REHACER) else pd.DataFrame()
hechos = set(previos_res["mes"]) if len(previos_res) else set()
filas = previos_res.to_dict("records")

for train, test, ini, fin in M.ventanas_mensuales(tabla, desde=DESDE, min_dias_train=MIN_DIAS_TRAIN):
    mes = f"{ini:%Y-%m}"
    if (SOLO_MESES and mes not in SOLO_MESES) or mes in hechos:
        continue
    preds = {"Promedio histórico": M.baseline_promedio(train, test),
             "Media 4 semanas": M.baseline_media_4sem(test)}
    n_arb = M.elegir_arboles(train, params)
    preds["XGBoost"] = np.clip(M.entrenar_xgb(train, params, n_arb).predict(test[M.FEATURES]), 0, None)
    for nombre, p in preds.items():
        t = por_hora(test, p)
        t.insert(0, "modelo", nombre)
        t.insert(0, "mes", mes)
        filas += t.to_dict("records")
        r = M.metricas(test[M.TARGET], p)
        print(f"{mes} {nombre:<20s} RMSE {r['RMSE']:.3f}  sesgo {r['Sesgo']:+.3f}")
    pd.DataFrame(filas).round(4).to_csv(SALIDA, index=False)      # se guarda después de cada mes

res = pd.DataFrame(filas)
print("Meses en el archivo:", sorted(res["mes"].unique()))

### Comprobación

El sesgo total de cada mes (promediando todas las horas, ponderado por demanda) debe coincidir con el de `resultados_mensual.csv` de `06`.

In [ ]:
ruta06 = "../data/processed/resultados_mensual.csv"
if os.path.exists(ruta06):
    viejo = pd.read_csv(ruta06)
    viejo = viejo[viejo["modelo"].str.startswith("XGBoost")].set_index("mes")["Sesgo"]
    # sesgo total = promedio de las 24 horas ponderado por filas; cada hora tiene el mismo número de filas por mes,
    # así que basta el promedio simple de los sesgos por hora
    nuevo = res[res["modelo"] == "XGBoost"].groupby("mes")["sesgo"].mean()
    comp = pd.DataFrame({"sesgo en este notebook": nuevo, "sesgo en 06": viejo.reindex(nuevo.index)}).round(3)
    display(comp)
else:
    print("No hay resultados_mensual.csv para comparar.")

## 2. Resultados

In [ ]:
x = res[(res["modelo"] == "XGBoost") & res["hora"].isin(HORAS)]
rel = x.pivot(index="mes", columns="hora", values="sesgo_rel_%")
display(rel.round(1).style.set_caption("Sesgo de XGBoost como % de la demanda de cada hora (positivo = predice de más)"))

fig, ax = plt.subplots(figsize=(11, 3.6))
lim = float(np.nanmax(np.abs(rel.to_numpy())))
im = ax.imshow(rel.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
ax.set_xticks(range(len(rel.columns)), rel.columns)
ax.set_yticks(range(len(rel.index)), rel.index)
ax.set_xlabel("Hora del día")
ax.set_title("Sesgo de XGBoost por mes y hora (% de la demanda; rojo = sobreestima)")
fig.colorbar(im, ax=ax, label="% de la demanda")
fig.tight_layout()
fig.savefig("../figures/23_sesgo_hora_mes.png", dpi=150)
plt.show()

In [ ]:
# Resumen por franja: ¿la sobreestimación nocturna aparece todos los meses?
franjas = {"Pico de la mañana (8 a 9 h)": [8, 9], "Mediodía (11 a 13 h)": [11, 12, 13],
           "Pico de la tarde (17 a 18 h)": [17, 18], "Noche (20 a 22 h)": [20, 21, 22]}
filas_f = []
for modelo in ["Promedio histórico", "Media 4 semanas", "XGBoost"]:
    d = res[res["modelo"] == modelo]
    for franja, horas in franjas.items():
        dh = d[d["hora"].isin(horas)].groupby("mes")
        s = 100 * dh["sesgo"].mean() / dh["demanda"].mean()
        for mes, v in s.items():
            filas_f.append({"franja": franja, "modelo": modelo, "mes": mes, "sesgo_rel_%": v})
franja_df = pd.DataFrame(filas_f)
for modelo in ["XGBoost", "Promedio histórico"]:
    t = franja_df[franja_df["modelo"] == modelo].pivot(index="franja", columns="mes", values="sesgo_rel_%")
    t = t.loc[list(franjas)]
    t["Promedio"] = t.mean(axis=1)
    display(t.round(1).style.set_caption(f"{modelo}: sesgo por franja (% de la demanda de la franja)"))

fig, ax = plt.subplots(figsize=(9, 4))
for modelo in ["Promedio histórico", "Media 4 semanas", "XGBoost"]:
    d = franja_df[(franja_df["modelo"] == modelo) & (franja_df["franja"] == "Noche (20 a 22 h)")]
    ax.plot(d["mes"], d["sesgo_rel_%"], marker="o", label=modelo)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("Sesgo (% de la demanda de 20 a 22 h)")
ax.set_xlabel("Mes de prueba")
ax.set_title("Sesgo de la noche (20 a 22 h) por mes")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("../figures/24_sesgo_noche_por_mes.png", dpi=150)
plt.show()

In [ ]:
res.round(4).to_csv(SALIDA, index=False)
franja_df.round(3).to_csv("../data/processed/resultados_sesgo_franja_mes.csv", index=False)
print("Guardado:", os.path.basename(SALIDA), "y resultados_sesgo_franja_mes.csv")

## Lectura de resultados

*(Por completar después de correr los siete meses.)*

Preguntas que hay que responder:

1. ¿La sobreestimación de 20 a 22 h aparece en **todos** los meses o solo en agosto (el periodo de prueba de `09`)?
2. ¿Es de XGBoost o de todos los modelos? Si el promedio histórico y la media de 4 semanas también sobreestiman de noche, no es un defecto del modelo sino del patrón de demanda de esas horas.
3. ¿Es de tamaño parecido en los meses de vacaciones (julio y agosto) que en los demás?
4. ¿La subestimación del pico de la mañana (8 y 9 h) también se repite?

**Qué decidir con el resultado:**
- Si aparece **todos los meses**, hay un sesgo sistemático de la noche que se puede atacar con una variable o un ajuste por hora, y vale la pena probarlo.
- Si aparece **solo en algunos meses** (por ejemplo julio y agosto), es más probable que sea un cambio de comportamiento del periodo (vacaciones) que el modelo no sigue a tiempo; el calendario escolar de `03` iría en esa dirección.
- Si **no aparece** en los otros meses, el patrón de `09` era del periodo de prueba.

**Limitaciones:**
- Los hiperparámetros se eligieron con datos de junio y julio de 2026 (igual que en `06`).
- Las horas 6 a 23 h; de madrugada el sistema está cerrado y no se analiza.
- Esto describe el sesgo promedio; no dice si el error en una hora concreta importa para el rebalanceo.